# Validate results and Delta lineage

**Audience:** data engineers validating medallion architecture and AIDP lineage.

**Prerequisites:** the canonical lab assets, shared compute and five job parameters.

**Learning goals:** trace governed transformations, verify isolation, and inspect deterministic results.


In [ ]:
import re
from functools import reduce
from pyspark.sql import Window, functions as F

# oidlUtils is injected by AIDP Workbench; no import is required.
def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")
catalog_name = required_parameter("catalog_name")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != "telco_lineage":
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")
if catalog_name != f"{participant_key}_aidp":
    raise ValueError("Invalid participant catalog")
spark.conf.set("spark.aidp.lineage.enabled", "true")

layer_prefixes = {"landing": "01_landing", "bronze": "02_bronze", "silver": "03_silver", "gold": "04_gold"}

def table(layer, logical_name):
    return f"{catalog_name}.oci_{layer}.{participant_key}_{lab_id}_{logical_name}"

def location(layer, logical_name):
    return f"oci://{bucket_name}@{objectstorage_namespace}/{layer_prefixes[layer]}/users/{participant_key}/{lab_id}/{logical_name}/"

def write_delta(frame, layer, logical_name, _ddl):
    target = table(layer, logical_name)
    (frame.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target))
    actual = spark.table(target).count()
    assert actual == frame.count(), f"Delta count mismatch for {logical_name}"
    print(f"Delta {layer}.{logical_name}: {actual} rows")


## Transformation

Run this cell once. It is idempotent and checks its row-level contract.


In [ ]:
expected_counts = {'customer_master': 493, 'customer_addresses': 617, 'product_catalog': 15, 'prepaid_service': 645, 'postpaid_service': 398, 'home_service': 218, 'service_ownership': 1261, 'quality_issues': 30, 'customer_360': 493, 'customer_service_portfolio': 1261, 'geographic_service_summary': 12}
for logical_name, expected in expected_counts.items():
    layer = "gold" if logical_name in {"customer_360", "customer_service_portfolio", "geographic_service_summary"} else "silver"
    actual = spark.table(table(layer, logical_name)).count()
    assert actual == expected, f"{layer}.{logical_name} expected {expected}, got {actual}"

for layer, logical_names in {'landing': ['crm_customers', 'crm_addresses', 'product_catalog', 'prepaid_lines', 'prepaid_recharges', 'postpaid_accounts', 'postpaid_lines', 'postpaid_invoices', 'home_services', 'home_installations'], 'bronze': ['crm_customers', 'crm_addresses', 'product_catalog', 'prepaid_lines', 'prepaid_recharges', 'postpaid_accounts', 'postpaid_lines', 'postpaid_invoices', 'home_services', 'home_installations'], 'silver': ['customer_master', 'customer_addresses', 'product_catalog', 'prepaid_service', 'postpaid_service', 'home_service', 'service_ownership', 'quality_issues'], 'gold': ['customer_360', 'customer_service_portfolio', 'geographic_service_summary']}.items():
    for logical_name in logical_names:
        details = spark.sql(f"DESCRIBE FORMATTED {table(layer, logical_name)}")
        formatted = {
            str(row["col_name"]).strip().lower(): str(row["data_type"]).strip().lower()
            for row in details.collect()
        }
        expected_provider = "delta"
        assert formatted.get("provider") == expected_provider, f"{table(layer, logical_name)} provider mismatch: {formatted.get('provider')}"
        assert formatted.get("type") == "managed", f"{table(layer, logical_name)} must be managed: {formatted.get('type')}"
        if layer != "landing":
            assert spark.sql(f"DESCRIBE HISTORY {table(layer, logical_name)}").count() >= 1

portfolio = spark.table(table("gold", "customer_service_portfolio"))
service_mix = {row["service_type"]: row["count"] for row in portfolio.groupBy("service_type").count().collect()}
assert service_mix == {"PREPAID": 645, "POSTPAID": 398, "HOME": 218}
customer_total = spark.table(table("gold", "customer_360")).agg(F.sum("monthly_value_total").alias("value")).first()["value"]
portfolio_total = portfolio.agg(F.sum("monthly_value").alias("value")).first()["value"]
geographic_total = spark.table(table("gold", "geographic_service_summary")).agg(F.sum("monthly_value_total").alias("value")).first()["value"]
assert abs(float(customer_total) - float(portfolio_total)) < 0.01
assert abs(float(customer_total) - float(geographic_total)) < 0.01
assert spark.conf.get("spark.aidp.lineage.enabled", "true").lower() == "true"
print("Telco Customer 360 validated: 31 managed tables, 30 quality issues, governed medallion lineage")


## Exercise and common pitfall

**Exercise:** follow one customer or service identifier into the next task and explain every derived column.

**Answer scaffold:** identify the source table, join key, transformation and target column.

**Pitfall:** never replace the job parameters with participant-specific literals; doing so breaks canonical hashes and isolation.

**Extension:** inspect the resulting entity and column lineage in Master Catalog.
